In [ ]:
import os
import time
import shutil
import ee
import numpy as np
import pandas as pd
from google.colab import files

try:
    OUTPUT_DIR = 'SoilMate_Export'
    MASTER_DIR = os.path.join(OUTPUT_DIR, 'Master Data')
    SAMPLE_DIR = os.path.join(OUTPUT_DIR, 'Sample Data')
    os.makedirs(MASTER_DIR, exist_ok=True)
    os.makedirs(SAMPLE_DIR, exist_ok=True)

    MY_PROJECT_ID = 'proven-cosmos-479318-j9'
    try:
        ee.Initialize(project=MY_PROJECT_ID)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=MY_PROJECT_ID)

    print("✅ Cell 1 succeeded: folders created, GEE connection established.")
except Exception as e:
    print(f"❌ Cell 1 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    START_DATE = '2000-01-01'
    END_DATE   = '2026-01-01'     # exclusive, must be a year start
    BUFFER_M   = 5000             # ROI radius in meters
    CROP_THRESHOLD    = 0.5       # keep coarse pixels that are more than 50% cropland
    MIN_CROP_FRACTION = 0.20      # warn if a site has less cropland than this

    # Approximate town centers in and around the Cukurova plain (edit freely).
    # Cell 3 prints the cropland share of each site so you can check them.
    SITES_RAW = {
        'Adana_Center': (37.0000, 35.3213),
        'Ceyhan':       (37.0247, 35.8124),
        'Imamoglu':     (37.2656, 35.6592),
        'Kozan':        (37.4553, 35.8167),
        'Tarsus':       (36.9164, 34.8920),
        'Osmaniye':     (37.0746, 36.2464),
    }
    SITES = {
        name: {'lat': lat, 'lon': lon,
               'roi': ee.Geometry.Point([lon, lat]).buffer(BUFFER_M)}
        for name, (lat, lon) in SITES_RAW.items()
    }

    # ESA WorldCover 2021: class 40 = cropland (10 m)
    cropland = ee.ImageCollection("ESA/WorldCover/v200").first().select('Map').eq(40)

    def build_crop_mask(projection):
        """Cropland share of every coarse pixel (computed from 10 m data).
        Returns a mask that keeps pixels above CROP_THRESHOLD."""
        share = cropland.reduceResolution(
            reducer=ee.Reducer.mean(), bestEffort=True, maxPixels=65536
        ).reproject(projection)
        return share.gt(CROP_THRESHOLD)

    def extract_time_series(collection, bands, scale, roi, chunk_years=4, prep=None):
        """Extract a time series from an ImageCollection for one ROI.
        Splits the range into chunks to stay under GEE's 5000-element limit.
        prep: optional function applied to each image (masking, band math)."""
        start_year = int(START_DATE[:4])
        end_year   = int(END_DATE[:4])

        def reduce_region(image):
            stats = image.select(bands).reduceRegion(
                reducer=ee.Reducer.mean(), geometry=roi, scale=scale, maxPixels=1e13)
            return ee.Feature(None, stats).set('date', image.date().format('YYYY-MM-dd'))

        rows = []
        for year in range(start_year, end_year, chunk_years):
            chunk_start = f"{year}-01-01"
            chunk_end   = f"{min(year + chunk_years, end_year)}-01-01"
            chunk_col   = collection.filterDate(chunk_start, chunk_end)

            try:
                size = chunk_col.size().getInfo()
            except Exception:
                size = 0
            if size == 0:
                continue

            if prep is not None:
                chunk_col = chunk_col.map(prep)

            print(f"    {chunk_start} to {chunk_end} ({size} images)")
            features = chunk_col.map(reduce_region).getInfo()['features']
            rows.extend(f['properties'] for f in features)

        df = pd.DataFrame(rows)
        if df.empty:
            return df
        df['date'] = pd.to_datetime(df['date'])
        df = df.drop_duplicates(subset=['date']).set_index('date').sort_index()
        return df.reindex(columns=bands)     # keeps all columns even if a band was fully masked

    def run_per_site(label, fetch_fn, store):
        """Run fetch_fn(site_name, roi) for every site and report the overall status."""
        failed = []
        for name, info in SITES.items():
            print(f"  [{name}]")
            t0 = time.time()
            try:
                store[name] = fetch_fn(name, info['roi'])
                print(f"  done in {time.time() - t0:.0f}s")
            except Exception as e:
                failed.append(name)
                print(f"  ❌ {name} failed: {type(e).__name__}: {e}")
        if not failed:
            print(f"✅ {label} succeeded for all {len(SITES)} sites.")
        elif len(failed) < len(SITES):
            print(f"⚠️ {label} finished with failures at: {failed}")
        else:
            print(f"❌ {label} failed for all sites.")

    # Storage for every data source (one entry per site)
    era5_data, chirps_data, vi_data, lst_data, ndwi_data = {}, {}, {}, {}, {}
    static_data, site_info = {}, {}

    print(f"✅ Cell 2 succeeded: {len(SITES)} sites and helper functions defined.")
except Exception as e:
    print(f"❌ Cell 2 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Scanning the region for cropland-rich sites...")

    lat_range = np.arange(36.5, 37.5, 0.1)
    lon_range = np.arange(34.6, 36.4, 0.1)

    grid = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([float(lon), float(lat)]).buffer(BUFFER_M),
                   {'lat': round(float(lat), 2), 'lon': round(float(lon), 2)})
        for lat in lat_range for lon in lon_range
    ])

    result = cropland.reduceRegions(
        collection=grid, reducer=ee.Reducer.mean(), scale=100).getInfo()

    cand = pd.DataFrame([f['properties'] for f in result['features']])
    cand = cand.rename(columns={'mean': 'cropland_share'}).dropna()
    cand = cand.sort_values('cropland_share', ascending=False)

    # Greedy pick: share >= 50% and at least 0.2 degrees (~22 km) apart,
    # so sites do not fall into the same 11 km ERA5 pixel
    selected = []
    for _, r in cand.iterrows():
        if r['cropland_share'] < 0.5:
            break
        if all(max(abs(r['lat'] - s['lat']), abs(r['lon'] - s['lon'])) >= 0.2 for s in selected):
            selected.append(r)
        if len(selected) == 8:
            break

    print(f"Candidates with >= 50% cropland: {(cand['cropland_share'] >= 0.5).sum()} of {len(cand)} grid points")
    print("\nPaste these into SITES_RAW in Cell 2:\n")
    for i, r in enumerate(selected, start=1):
        print(f"        'Site_{i}': ({r['lat']}, {r['lon']}),   # cropland {r['cropland_share']:.1%}")

    print(f"\n✅ Cell 3b succeeded: {len(selected)} candidate sites selected.")
except Exception as e:
    print(f"❌ Cell 3b failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Fetching ERA5-Land...")
    era5_bands = [
        'temperature_2m', 'dewpoint_temperature_2m', 'total_precipitation_sum',
        'surface_solar_radiation_downwards_sum', 'total_evaporation_sum', 'potential_evaporation_sum',
        'volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2',
        'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4'
    ]

    def fetch_era5(name, roi):
        col = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").filterBounds(roi)
        df = extract_time_series(col, era5_bands, 11000, roi, chunk_years=5)
        if df.empty:
            raise ValueError("ERA5 returned no data.")
        df['temperature_2m']          -= 273.15
        df['dewpoint_temperature_2m'] -= 273.15
        for c in ['total_precipitation_sum', 'total_evaporation_sum', 'potential_evaporation_sum']:
            df[c] *= 1000                                    # m -> mm
        df['surface_solar_radiation_downwards_sum'] /= 86400  # J/m2 -> W/m2
        return df

    run_per_site("Cell 4 (ERA5-Land)", fetch_era5, era5_data)

    if era5_data:   # sanity check: ERA5 evaporation is usually negative
        first = next(iter(era5_data.values()))
        print("Mean total_evaporation_sum (mm/day):", round(first['total_evaporation_sum'].mean(), 3))
except Exception as e:
    print(f"❌ Cell 4 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Fetching CHIRPS...")

    def fetch_chirps(name, roi):
        col = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterBounds(roi)
        df = extract_time_series(col, ['precipitation'], 5566, roi, chunk_years=5)
        if df.empty:
            raise ValueError("CHIRPS returned no data.")
        return df.rename(columns={'precipitation': 'chirps_precipitation'})   # mm/day

    run_per_site("Cell 5 (CHIRPS)", fetch_chirps, chirps_data)
except Exception as e:
    print(f"❌ Cell 5 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Fetching MODIS NDVI/EVI...")

    def fetch_vi(name, roi):
        col  = ee.ImageCollection("MODIS/061/MOD13Q1").filterBounds(roi)
        mask = build_crop_mask(col.first().select('NDVI').projection())

        def prep(img):
            good = img.select('SummaryQA').lte(1)     # 0 = good, 1 = marginal
            return img.updateMask(good).updateMask(mask).copyProperties(img, ['system:time_start'])

        df = extract_time_series(col, ['NDVI', 'EVI'], 250, roi, chunk_years=26, prep=prep)
        if df.empty:
            raise ValueError("MODIS NDVI/EVI returned no data.")
        df[['NDVI', 'EVI']] *= 0.0001                 # scale factor
        return df

    run_per_site("Cell 6 (MODIS NDVI/EVI)", fetch_vi, vi_data)
except Exception as e:
    print(f"❌ Cell 6 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Fetching MODIS LST...")

    def fetch_lst(name, roi):
        col  = ee.ImageCollection("MODIS/061/MOD11A2").filterBounds(roi)
        mask = build_crop_mask(col.first().select('LST_Day_1km').projection())

        def prep(img):
            return img.updateMask(mask).copyProperties(img, ['system:time_start'])

        df = extract_time_series(col, ['LST_Day_1km', 'LST_Night_1km'], 1000, roi,
                                 chunk_years=26, prep=prep)
        if df.empty:
            raise ValueError("MODIS LST returned no data.")
        df['LST_Day_1km']   = df['LST_Day_1km']   * 0.02 - 273.15    # Kelvin -> Celsius
        df['LST_Night_1km'] = df['LST_Night_1km'] * 0.02 - 273.15
        return df

    run_per_site("Cell 7 (MODIS LST)", fetch_lst, lst_data)
except Exception as e:
    print(f"❌ Cell 7 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Computing MODIS NDWI...")

    def fetch_ndwi(name, roi):
        col  = ee.ImageCollection("MODIS/061/MOD09A1").filterBounds(roi)
        mask = build_crop_mask(col.first().select('sur_refl_b02').projection())

        def prep(img):
            clear = img.select('StateQA').bitwiseAnd(3).eq(0)                                   # cloud-free
            ndwi  = img.normalizedDifference(['sur_refl_b02', 'sur_refl_b06']).rename('NDWI')   # NIR vs SWIR
            return ndwi.updateMask(clear).updateMask(mask).copyProperties(img, ['system:time_start'])

        df = extract_time_series(col, ['NDWI'], 500, roi, chunk_years=26, prep=prep)
        if df.empty:
            raise ValueError("NDWI returned no data.")
        return df

    run_per_site("Cell 8 (NDWI)", fetch_ndwi, ndwi_data)
except Exception as e:
    print(f"❌ Cell 8 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    print("Fetching static data...")

    def fetch_static(name, roi):
        elevation = ee.Image("USGS/SRTMGL1_003").select('elevation')
        slope = ee.Terrain.slope(elevation).rename('slope')
        clay  = ee.Image("projects/soilgrids-isric/clay_mean").select(0).rename('clay')
        sand  = ee.Image("projects/soilgrids-isric/sand_mean").select(0).rename('sand')
        phh2o = ee.Image("projects/soilgrids-isric/phh2o_mean").select(0).rename('phh2o')

        stats = ee.Image([elevation, slope, clay, sand, phh2o]).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=roi, scale=250, maxPixels=1e13).getInfo()
        if stats.get('phh2o') is not None:
            stats['phh2o'] = stats['phh2o'] / 10          # pH scale correction
        stats['cropland_fraction'] = site_info.get(name, np.nan)
        print(f"  {stats}")
        return stats

    run_per_site("Cell 9 (static data)", fetch_static, static_data)
except Exception as e:
    print(f"❌ Cell 9 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    master_frames = []
    for name in SITES:
        sources = [era5_data.get(name), chirps_data.get(name), vi_data.get(name),
                   lst_data.get(name), ndwi_data.get(name)]
        if any(s is None for s in sources) or name not in static_data:
            print(f"⚠️ Skipping {name}: at least one data source is missing.")
            continue

        common_start = max(s.index.min() for s in sources)    # first day all sources have data
        df = sources[0].astype(float).join(sources[1:], how='outer')
        df = df.interpolate(method='time')                    # time-based, handles irregular intervals
        df = df.loc[common_start:].ffill()                    # no fake back-fill, only the tail is filled

        all_nan = [c for c in df.columns if df[c].isna().all()]
        if all_nan:
            print(f"⚠️ Skipping {name}: columns with no valid data: {all_nan}")
            continue

        for key, value in static_data[name].items():
            df[key] = float(value) if value is not None else np.nan
        df.insert(0, 'Site', name)
        master_frames.append(df)
        print(f"  {name}: {len(df)} rows, {df.index.min().date()} -> {df.index.max().date()}")

    if not master_frames:
        raise ValueError("No site produced a complete dataset.")

    master = pd.concat(master_frames).round(4)
    master.index.name = 'Date'

    master_path = os.path.join(MASTER_DIR, f"SoilMate_MasterData_multisite_{START_DATE}_to_{END_DATE}.csv")
    master.to_csv(master_path)

    print(f"Sites: {master['Site'].nunique()}, Rows: {len(master)}, Columns: {len(master.columns)}")
    print(f"✅ Cell 10 succeeded: master data saved to {master_path}")
except Exception as e:
    print(f"❌ Cell 10 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    df_daily = pd.read_csv(master_path, parse_dates=['Date'], index_col='Date')

    grouping_days = [3, 5, 7, 10, 14, 15, 20, 30]
    sum_cols = ['total_precipitation_sum', 'total_evaporation_sum',
                'potential_evaporation_sum', 'chirps_precipitation']   # only cumulative variables are summed

    for days in grouping_days:
        rule  = 'MS' if days == 30 else f'{days}D'      # 30 = calendar month
        label = "Monthly (calendar month)" if days == 30 else f"{days}-day"

        parts = []
        for site, g in df_daily.groupby('Site'):
            g = g.drop(columns='Site')
            agg = {c: ('sum' if c in sum_cols else 'mean') for c in g.columns}

            counts   = g.resample(rule).size()
            expected = counts.index.days_in_month if days == 30 else days
            is_full  = counts.values == np.asarray(expected)          # drop incomplete periods

            grouped = g.resample(rule).agg(agg).round(4)[is_full].dropna()

            # Calendar features from the middle of each period
            mid = grouped.index + pd.to_timedelta(days / 2, unit='D')
            grouped['doy_sin'] = np.sin(2 * np.pi * mid.dayofyear / 365.25).round(4)
            grouped['doy_cos'] = np.cos(2 * np.pi * mid.dayofyear / 365.25).round(4)

            grouped.insert(0, 'Site', site)
            parts.append(grouped)

        df_grouped = pd.concat(parts)
        df_grouped.index.name = 'Date'
        df_grouped.to_csv(os.path.join(SAMPLE_DIR, f"grouped_{days}_days.csv"))
        print(f"  {label}: {len(df_grouped)} rows ({df_grouped['Site'].nunique()} sites)")

    print(f"✅ Cell 11 succeeded: {len(grouping_days)} grouped datasets saved.")
except Exception as e:
    print(f"❌ Cell 11 failed: {type(e).__name__}: {e}")

In [ ]:
try:
    zip_name = 'SoilMate_Export'
    shutil.make_archive(zip_name, 'zip', root_dir='.', base_dir=OUTPUT_DIR)
    files.download(f"{zip_name}.zip")

    print(f"✅ Cell 12 succeeded: {zip_name}.zip created, download started.")
except Exception as e:
    print(f"❌ Cell 12 failed: {type(e).__name__}: {e}")